# Laya: architecture tests (experiments 1, 2, 2b, 3b, 4 and 6)
Notebook settings: **Accelerator = GPU T4 x2**, **Internet = On**.
First run with `QUICK = True` (a short smoke test). If everything passes, set it to `False` and use
*Save Version → Save & Run All* to run in the background. Everything is saved to `/kaggle/working/laya_bench`.
Experiment 3 (a 21-pair pilot) is off by default; the minimal-edit results come from Experiment 3b.

In [ ]:
!pip install -q "laya==0.3.21" umap-learn

In [ ]:
# 0. Configuration
import os
os.environ["USE_TF"] = "0"            # avoids a laya.load() hang when TensorFlow is installed
os.environ["LAYA_CUDA_AMP"] = "fp16"  # the T4 forces fp16 anyway; set explicitly so it shows in the logs

import json, time, re, sys, platform, logging, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

QUICK = True   # True = few examples, only to validate the pipeline

CFG = dict(
    seed=42,
    exp1_n_options=[2, 10, 50, 150, 500], exp1_n_examples=100, exp1_latency_repeats=30,
    exp2_n_options=[4, 10, 20, 50, 77],   exp2_n_examples=100, exp2_n_perms=5,
    exp4_per_intent=20, exp6_per_intent=25,
    fit_state_room=160,   # tokens reserved for the state in the "fit" regime
    max_len_cap=8192,     # ModernBERT limit
)
if QUICK:
    CFG.update(exp1_n_examples=5, exp1_latency_repeats=5, exp2_n_examples=5, exp2_n_perms=3,
               exp4_per_intent=2, exp6_per_intent=4)
RUN = dict(exp1=True, exp2=True, exp2b=True, exp3=False, exp3b=True, exp4=True, exp6=True)  # exp3 = 21-pair pilot, superseded by 3b

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT = ROOT / "laya_bench"
for d in ["results", "logs", "figures"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d-%H%M%S") + ("-quick" if QUICK else "")
SFX = "_quick" if QUICK else ""

log = logging.getLogger("laya_bench"); log.setLevel(logging.INFO); log.handlers.clear()
for h in [logging.FileHandler(OUT / "logs" / f"run_{RUN_ID}.log"), logging.StreamHandler(sys.stdout)]:
    h.setFormatter(logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")); log.addHandler(h)
log.info(f"run_id={RUN_ID} out={OUT}")

In [ ]:
# 1. Model, metadata and sanity check
import laya, transformers, datasets
from laya.common import build_sequence, render_options, serialize_state, encode_text, QTYPES
from laya.agent import Agent

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
t0 = time.perf_counter()
agent = laya.load("convaiinnovations/laya", device=DEVICE)
TOK = agent.tok
DEFAULT_BUDGET = dict(max_len=agent.cfg.get("max_len", 512), head_max_len=agent.cfg.get("head_max_len", 192))

meta = dict(run_id=RUN_ID, quick=QUICK, cfg=CFG, python=sys.version.split()[0], platform=platform.platform(),
            torch=torch.__version__, transformers=transformers.__version__, datasets=datasets.__version__,
            laya=laya.__version__, device=DEVICE,
            gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
            precision=str(agent.dtype) if agent.amp_enabled else "fp32",
            default_budget=DEFAULT_BUDGET, load_seconds=round(time.perf_counter() - t0, 1))

def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

def timed_predict(state, questions, **budget):
    sync(); t = time.perf_counter()
    res = agent.predict(state, questions, **budget)
    sync(); return res, (time.perf_counter() - t) * 1000

sanity_q = {"department": {"type": "choice", "instructions": "Which department should handle this?",
            "criteria": {"billing": "invoices, payments, refunds", "technical": "bugs, outages, system errors",
                         "other": "everything else"}}}
sanity_state = "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."
for _ in range(3): agent.predict(sanity_state, sanity_q)
lat = [timed_predict(sanity_state, sanity_q)[1] for _ in range(20)]
ans = agent.predict(sanity_state, sanity_q)["answers"]["department"]
meta["sanity"] = dict(choice=ans["choice"], probabilities=ans["probabilities"], latency_ms_median=float(np.median(lat)))
(OUT / "results" / f"run_meta_{RUN_ID}.json").write_text(json.dumps(meta, indent=2, default=str))
log.info(f"meta: {json.dumps({k: meta[k] for k in ['gpu', 'precision', 'laya', 'transformers', 'default_budget']})}")
log.info(f"sanity: {meta['sanity']}")
assert ans["choice"] == "billing", "Sanity check failed: the README example did not return 'billing'"

In [ ]:
# 2. Utilities: resumable JSONL logs, sequence statistics, "fit" budget
def jdefault(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return str(o)

class JsonlLog:
    """One line per prediction. Keys already written are skipped (resume within a session)."""
    def __init__(self, name):
        self.path = OUT / "results" / f"{name}{'_quick' if QUICK else ''}.jsonl"  # the smoke test never mixes with the real run
        self.done = set()
        if self.path.exists():
            for line in self.path.open():
                try: self.done.add(json.loads(line)["key"])
                except Exception: pass
    def write(self, rec):
        rec = {"run_id": RUN_ID, **rec}
        with self.path.open("a") as f: f.write(json.dumps(rec, default=jdefault) + "\n")
        self.done.add(rec["key"])
    def df(self):
        return pd.read_json(self.path, lines=True) if self.path.exists() else pd.DataFrame()

def humanize(label):
    return re.sub(r"^in:", "", str(label).strip().lower()).replace("_", " ").strip()

def choice_q(criteria, instructions):
    return {"q": {"type": "choice", "instructions": instructions, "criteria": criteria}}

def ntok(text):
    return len(encode_text(TOK, text, add_special_tokens=False)["input_ids"])

def seq_stats(state, questions, max_len, head_max_len):
    """Rebuilds Laya's input sequence and measures what the token budget cut."""
    q = Agent._to_internal(questions["q"])
    ids, markers, st = build_sequence(TOK, state, q, max_len, head_max_len, return_stats=True)
    seps = [i for i, t in enumerate(ids) if t == TOK.sep_token_id]
    kept = (seps[2] if len(seps) >= 3 else len(ids)) - (seps[1] + 1) if len(seps) >= 2 else 0
    return dict(seq_len=len(ids), n_markers=len(markers), options_distinct=st["options_distinct"],
                tokens_per_option=st["tokens_per_option"], state_tokens_total=ntok(serialize_state(state)),
                state_tokens_kept=max(0, kept))

def fit_budget(state, questions):
    """A budget in which every option fits whole (up to 48 tokens) and the state is not cut."""
    q = Agent._to_internal(questions["q"])
    opts = sum(1 + min(48, ntok(" " + o)) for o in render_options(q))
    head = ntok("%s question: %s" % (q["t"], q["ins"]))
    head_max_len = opts + head + 16
    need = 4 + head + opts + max(CFG["fit_state_room"], ntok(serialize_state(state)))
    max_len = min(CFG["max_len_cap"], int(np.ceil(need / 64) * 64))
    return dict(max_len=max_len, head_max_len=min(head_max_len, max_len - 8))

def budget_for(regime, state, questions):
    return dict(DEFAULT_BUDGET) if regime == "default" else fit_budget(state, questions)

def tv(p, q):
    keys = set(p) | set(q)
    return 0.5 * sum(abs(p.get(k, 0) - q.get(k, 0)) for k in keys)

def savefig(name):
    plt.tight_layout(); plt.savefig(OUT / "figures" / f"{name}{SFX}.png", dpi=160); plt.show()

In [ ]:
# 3. Data: banking77, clinc150 and the label pool for up to 500 options
from datasets import load_dataset
b77 = load_dataset("mteb/banking77", split="test").to_pandas()[["text", "label_text"]]
B77_LABELS = sorted(b77["label_text"].unique())
log.info(f"banking77: {len(b77)} texts, {len(B77_LABELS)} intents")

POOL, seen, POOL_SOURCES = [], set(), {}
def add_to_pool(labels, source):
    n0 = len(POOL)
    for lab in labels:
        h = humanize(lab)
        if h and h not in seen and h != "oos":
            seen.add(h); POOL.append((h, source))
    POOL_SOURCES[source] = len(POOL) - n0

add_to_pool(B77_LABELS, "banking77")
try:
    clinc = load_dataset("clinc/clinc_oos", "plus", split="test")
    add_to_pool(clinc.features["intent"].names, "clinc150")
except Exception as e:
    log.info(f"clinc150 failed: {e!r}")
for repo, conf in [("mteb/amazon_massive_intent", "en"), ("mteb/mtop_intent", "en")]:
    try:
        ds = load_dataset(repo, conf, split="test")
        col = "label_text" if "label_text" in ds.column_names else "label"
        add_to_pool(sorted(set(ds[col])), repo)
    except Exception as e:
        log.info(f"{repo} failed: {e!r}")
log.info(f"label pool: {len(POOL)} unique, by source: {POOL_SOURCES}")
json.dump({"pool_size": len(POOL), "by_source": POOL_SOURCES, "labels": POOL},
          open(OUT / "results" / "label_pool.json", "w"), indent=1)

INSTR = "What does the customer want? Pick the intent that matches the message."

def option_set(gold_h, n, rng):
    """Correct answer + (n-1) distractors: banking77 first, then the other sources."""
    b = [h for h, s in POOL if s == "banking77" and h != gold_h]
    rest = [h for h, s in POOL if s != "banking77"]
    rng.shuffle(b); rng.shuffle(rest)
    opts = [gold_h] + (b + rest)[: n - 1]
    rng.shuffle(opts)
    return opts

def sample_b77(n, seed):
    return b77.sample(n=min(n, len(b77)), random_state=seed).reset_index(drop=True)

In [ ]:
# 4. Experiment 1: scaling with the number of options
if RUN["exp1"]:
    L1, LB = JsonlLog("exp1_scaling"), JsonlLog("exp1_latency_bench")
    ex = sample_b77(CFG["exp1_n_examples"], CFG["seed"])
    for regime in ["default", "fit"]:
        for n_req in CFG["exp1_n_options"]:
            n = min(n_req, len(POOL))
            if n < n_req: log.info(f"exp1: the pool only has {len(POOL)} labels; N={n_req} becomes {n}")
            # latency micro-benchmark: same state, same options, several repetitions
            rng = np.random.default_rng(CFG["seed"] + n)
            s0 = ex.loc[0, "text"]; qs0 = choice_q({o: None for o in option_set(humanize(ex.loc[0, "label_text"]), n, rng)}, INSTR)
            bud0 = budget_for(regime, s0, qs0)
            try:
                for _ in range(3): agent.predict(s0, qs0, **bud0)
                for r in range(CFG["exp1_latency_repeats"]):
                    k = f"{regime}|{n_req}|{r}"
                    if k in LB.done: continue
                    _, ms = timed_predict(s0, qs0, **bud0)
                    LB.write(dict(key=k, regime=regime, n_req=n_req, n=n, rep=r, latency_ms=ms, **bud0,
                                  **seq_stats(s0, qs0, **bud0), status="ok"))
            except Exception as e:
                LB.write(dict(key=f"{regime}|{n_req}|error", regime=regime, n_req=n_req, n=n, rep=-1, **bud0,
                              status=type(e).__name__, error=str(e)[:200]))
                if DEVICE == "cuda": torch.cuda.empty_cache()
            # accuracy and statistics over many texts
            for i, row in ex.iterrows():
                k = f"{regime}|{n_req}|{i}"
                if k in L1.done: continue
                gold = humanize(row["label_text"])
                rng = np.random.default_rng(CFG["seed"] * 1000 + i * 7 + n)
                qs = choice_q({o: None for o in option_set(gold, n, rng)}, INSTR)
                bud = budget_for(regime, row["text"], qs)
                rec = dict(key=k, regime=regime, n_req=n_req, n=n, idx=i, gold=gold, **bud)
                try:
                    rec.update(seq_stats(row["text"], qs, **bud))
                    res, ms = timed_predict(row["text"], qs, **bud)
                    a = res["answers"]["q"]; p = a["probabilities"]
                    top5 = dict(sorted(p.items(), key=lambda kv: -kv[1])[:5])
                    rec.update(pred=a["choice"], correct=a["choice"] == gold, p_gold=p.get(gold, 0.0),
                               latency_ms=ms, top5=top5, status="ok")
                except Exception as e:
                    rec.update(status=type(e).__name__, error=str(e)[:200])
                    if DEVICE == "cuda": torch.cuda.empty_cache()
                L1.write(rec)
            log.info(f"exp1 {regime} N={n_req} ok")

    d1, db = L1.df(), LB.df()
    ok = d1[d1.status == "ok"]
    s1 = d1.groupby(["regime", "n_req"]).agg(
        n_eff=("n", "first"), fail_rate=("status", lambda s: (s != "ok").mean())).join(
        ok.groupby(["regime", "n_req"]).agg(
            accuracy=("correct", "mean"), p_gold=("p_gold", "mean"),
            distinct_frac=("options_distinct", "mean"), tokens_per_option=("tokens_per_option", "mean"),
            seq_len=("seq_len", "mean"), state_kept=("state_tokens_kept", "mean"),
            state_total=("state_tokens_total", "mean")))
    dbo = db[db.status == "ok"]
    s1 = s1.join(dbo.groupby(["regime", "n_req"]).latency_ms.agg(
        lat_median="median", lat_p90=lambda x: np.percentile(x, 90)))
    s1["distinct_frac"] = s1["distinct_frac"] / s1["n_eff"]
    s1.to_csv(OUT / "results" / f"exp1_summary{SFX}.csv"); display(s1)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for regime, g in s1.reset_index().groupby("regime"):
        ax[0].plot(g.n_req, g.lat_median, "o-", label=regime)
        ax[1].plot(g.n_req, g.accuracy, "o-", label=regime)
        ax[2].plot(g.n_req, g.distinct_frac, "o-", label=regime)
    for a, t in zip(ax, ["median latency (ms)", "accuracy", "fraction of distinct options"]):
        a.set_xscale("log"); a.set_xlabel("number of options"); a.set_title(t); a.legend()
    savefig("exp1_scaling")

In [ ]:
# 5. Experiment 2: invariance to option order
if RUN["exp2"]:
    L2 = JsonlLog("exp2_order")
    ex = sample_b77(CFG["exp2_n_examples"], CFG["seed"] + 1)
    for regime in ["default", "fit"]:
        for n in CFG["exp2_n_options"]:
            for i, row in ex.iterrows():
                gold = humanize(row["label_text"])
                rng = np.random.default_rng(CFG["seed"] * 2000 + i * 11 + n)
                base = option_set(gold, n, rng)
                for pi in range(CFG["exp2_n_perms"]):
                    k = f"{regime}|{n}|{i}|{pi}"
                    if k in L2.done: continue
                    order = list(base) if pi == 0 else list(rng.permutation(base))
                    qs = choice_q({o: None for o in order}, INSTR)
                    bud = budget_for(regime, row["text"], qs)
                    rec = dict(key=k, regime=regime, n=n, idx=i, perm=pi, gold=gold,
                               gold_pos=order.index(gold), **bud)
                    try:
                        rec.update(seq_stats(row["text"], qs, **bud))
                        a = agent.predict(row["text"], qs, **bud)["answers"]["q"]
                        rec.update(pred=a["choice"], correct=a["choice"] == gold, probs=a["probabilities"],
                                   p_gold=a["probabilities"].get(gold, 0.0), status="ok")
                    except Exception as e:
                        rec.update(status=type(e).__name__, error=str(e)[:200])
                    L2.write(rec)
            log.info(f"exp2 {regime} N={n} ok")

    d2 = L2.df(); d2 = d2[d2.status == "ok"]
    rows = []
    for (regime, n, i), g in d2.groupby(["regime", "n", "idx"]):
        P = list(g.probs)
        rows.append(dict(regime=regime, n=n, idx=i, flipped=g.pred.nunique() > 1,
                         mean_tv=np.mean([tv(P[a], P[b]) for a in range(len(P)) for b in range(a + 1, len(P))]),
                         p_gold_std=g.p_gold.std(), accuracy=g.correct.mean()))
    s2 = pd.DataFrame(rows).groupby(["regime", "n"]).agg(
        flip_rate=("flipped", "mean"), mean_tv=("mean_tv", "mean"),
        p_gold_std=("p_gold_std", "mean"), accuracy=("accuracy", "mean"))
    # position bias: accuracy by the relative position of the correct answer
    d2["pos_bin"] = pd.cut(d2.gold_pos / (d2.n - 1).clip(lower=1), [-0.01, 0.25, 0.5, 0.75, 1.0],
                           labels=["first", "second", "third", "last"])
    pos = d2.groupby(["regime", "n", "pos_bin"], observed=True).correct.mean().unstack()
    s2.to_csv(OUT / "results" / f"exp2_summary{SFX}.csv"); pos.to_csv(OUT / "results" / f"exp2_position_bias{SFX}.csv")
    display(s2); display(pos)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for regime, g in s2.reset_index().groupby("regime"):
        ax[0].plot(g.n, g.flip_rate, "o-", label=regime); ax[1].plot(g.n, g.mean_tv, "o-", label=regime)
    for a, t in zip(ax, ["fraction of texts whose answer changes", "mean distance between distributions (TV)"]):
        a.set_xlabel("number of options"); a.set_title(t); a.legend()
    savefig("exp2_order")

In [ ]:
# 6. Experiment 3 (pilot): minimal-edit pairs
OPT = dict(
    dup=("duplicate charge", "the same purchase was billed more than once"),
    unk=("unknown charge", "the customer does not recognise a single charge"),
    blk=("card blocked", "the card is blocked after too many wrong PIN attempts"),
    pin=("pin question", "a general question about the PIN"),
    mis=("missing item", "fewer items were delivered than were paid for"),
    okd=("order complete", "the order was delivered exactly as paid"),
    can=("cancel", "the customer wants to end the subscription"),
    kep=("keep", "the customer wants to keep the subscription"),
    ndl=("delivery problem", "the order has not been delivered"),
    dlv=("delivered", "the order was delivered"),
    frd=("fraud report", "a payment the customer did not make"),
    pmy=("payment confirmed", "the customer confirms the payment is theirs"),
    bug=("bug", "the software is broken or not working"),
    fin=("working fine", "the software works as expected"),
    suf=("sufficient funds", "the balance covers the withdrawal"),
    ins=("insufficient funds", "the withdrawal is larger than the balance"),
    amm=("amount mismatch", "charged a different amount than the invoice"),
    amo=("amount correct", "charged exactly the invoice amount"),
    wrg=("wrong item", "received a different item or size than ordered"),
    rgt=("correct item", "received exactly the item and size ordered"),
    ont=("on time", "the customer will make the flight"),
    mss=("missed flight", "the customer will miss the flight"),
    inc=("incoming transfer", "the customer received money"),
    out=("outgoing transfer", "the customer sent money"),
    gav=("refund issued", "the writer gave money back to someone"),
    got=("refund received", "the writer got money back from someone"),
    lat=("late payment", "paying after the deadline"),
    ear=("on-time payment", "paying before the deadline"),
    tfl=("transfers failed", "transfers did not go through"),
    tok=("transfers ok", "transfers went through normally"),
)
OTHER = ("other", "anything else")
# (id, kind, category, text_a, text_b, options, correct_a, correct_b)
PAIRS = [
    ("c01", "change", "count", "I was charged once for my coffee and I don't know what it is.", "I was charged twice for my coffee and I don't know what it is.", ["dup", "unk"], "unk", "dup"),
    ("c02", "change", "count", "I entered the wrong PIN once.", "I entered the wrong PIN three times.", ["blk", "pin"], "pin", "blk"),
    ("c03", "change", "count", "I paid for one ticket and received one ticket.", "I paid for two tickets and received one ticket.", ["mis", "okd"], "okd", "mis"),
    ("c04", "change", "negation", "I want to cancel my subscription.", "I do not want to cancel my subscription.", ["can", "kep"], "can", "kep"),
    ("c05", "change", "negation", "The package arrived.", "The package never arrived.", ["ndl", "dlv"], "dlv", "ndl"),
    ("c06", "change", "negation", "I recognise this payment on my statement.", "I don't recognise this payment on my statement.", ["frd", "pmy"], "pmy", "frd"),
    ("c07", "change", "negation", "The app works after the update.", "The app no longer works after the update.", ["bug", "fin"], "fin", "bug"),
    ("c08", "change", "numbers", "My balance is $500 and I want to withdraw $200.", "My balance is $200 and I want to withdraw $500.", ["suf", "ins"], "suf", "ins"),
    ("c09", "change", "numbers", "The invoice says $120 and I was charged $120.", "The invoice says $120 and I was charged $210.", ["amm", "amo"], "amo", "amm"),
    ("c10", "change", "numbers", "I ordered size 42 and received size 42.", "I ordered size 42 and received size 24.", ["wrg", "rgt"], "rgt", "wrg"),
    ("c11", "change", "numbers", "My flight leaves at 9 and I will get to the airport at 7.", "My flight leaves at 7 and I will get to the airport at 9.", ["ont", "mss"], "ont", "mss"),
    ("c12", "change", "roles", "The bank sent money to me.", "I sent money to the bank.", ["inc", "out"], "inc", "out"),
    ("c13", "change", "roles", "I refunded the customer.", "The customer refunded me.", ["gav", "got"], "gav", "got"),
    ("c14", "change", "time", "I want to pay before the due date.", "I want to pay after the due date.", ["lat", "ear"], "ear", "lat"),
    ("c15", "change", "quantifier", "All of my transfers failed today.", "None of my transfers failed today.", ["tfl", "tok"], "tfl", "tok"),
    ("s01", "same", "paraphrase", "I was charged twice for my coffee and I don't know what it is.", "I was charged two times for my coffee and I don't know what it is.", ["dup", "unk"], "dup", "dup"),
    ("s02", "same", "paraphrase", "I do not want to cancel my subscription.", "I don't want to cancel my subscription.", ["can", "kep"], "kep", "kep"),
    ("s03", "same", "paraphrase", "My balance is $200 and I want to withdraw $500.", "My balance is 200 dollars and I want to withdraw 500 dollars.", ["suf", "ins"], "ins", "ins"),
    ("s04", "same", "paraphrase", "The package never arrived.", "The parcel never arrived.", ["ndl", "dlv"], "ndl", "ndl"),
    ("s05", "same", "paraphrase", "I don't recognise this payment on my statement.", "I do not recognize this payment on my statement.", ["frd", "pmy"], "frd", "frd"),
    ("s06", "same", "paraphrase", "I sent money to the bank.", "I transferred money to the bank.", ["inc", "out"], "out", "out"),
]
INSTR3 = "Which option best describes the customer's message?"

if RUN["exp3"]:
    L3 = JsonlLog("exp3_minimal_pairs")
    for pid, kind, cat, ta, tb, opts, ga, gb in PAIRS:
        for order_name, oo in [("fwd", opts + ["OTHER"]), ("rev", list(reversed(opts)) + ["OTHER"])]:
            k = f"{pid}|{order_name}"
            if k in L3.done: continue
            crit = {(OTHER if o == "OTHER" else OPT[o])[0]: (OTHER if o == "OTHER" else OPT[o])[1] for o in oo}
            qs = choice_q(crit, INSTR3)
            ga_n, gb_n = OPT[ga][0], OPT[gb][0]
            bud = fit_budget(tb, qs)
            A = agent.predict(ta, qs, **bud)["answers"]["q"]; B = agent.predict(tb, qs, **bud)["answers"]["q"]
            L3.write(dict(key=k, pair=pid, kind=kind, category=cat, order=order_name, text_a=ta, text_b=tb,
                          gold_a=ga_n, gold_b=gb_n, pred_a=A["choice"], pred_b=B["choice"],
                          correct_a=A["choice"] == ga_n, correct_b=B["choice"] == gb_n,
                          flipped=A["choice"] != B["choice"], tv=tv(A["probabilities"], B["probabilities"]),
                          p_gold_b_in_a=A["probabilities"][gb_n], p_gold_b_in_b=B["probabilities"][gb_n],
                          probs_a=A["probabilities"], probs_b=B["probabilities"], **bud))
    d3 = L3.df()
    d3["both_correct"] = d3.correct_a & d3.correct_b
    d3["shift_to_gold_b"] = d3.p_gold_b_in_b - d3.p_gold_b_in_a
    s3 = d3.groupby(["kind", "category"]).agg(pairs=("pair", "nunique"), both_correct=("both_correct", "mean"),
                                              flip_rate=("flipped", "mean"), mean_tv=("tv", "mean"),
                                              shift_to_gold_b=("shift_to_gold_b", "mean"))
    s3.to_csv(OUT / "results" / f"exp3_summary{SFX}.csv"); display(s3)
    display(d3[["pair", "order", "pred_a", "pred_b", "gold_a", "gold_b", "tv"]])

    g = d3[d3.order == "fwd"].sort_values(["kind", "category"])
    plt.figure(figsize=(10, 5))
    plt.barh(g.pair + " " + g.category, g.shift_to_gold_b, color=np.where(g.kind == "change", "C0", "C7"))
    plt.axvline(0, color="k", lw=.8); plt.xlabel("change in P(B's correct answer) when switching A → B")
    plt.title("minimal edits: how much the probability moves (grey = pairs that should not change)")
    savefig("exp3_minimal_pairs")

In [ ]:
# 7. Experiment 4: labels without meaning
INTENTS4 = {
    "activate_my_card": "how to activate a newly received card",
    "age_limit": "minimum age required to open or use an account",
    "atm_support": "which cash machines can be used and how",
    "card_swallowed": "a cash machine kept or retained the card",
    "change_pin": "how to change the card PIN code",
    "exchange_rate": "what currency exchange rate is applied",
    "lost_or_stolen_phone": "the phone with the banking app was lost or stolen",
    "terminate_account": "close or delete the account permanently",
    "top_up_limits": "maximum amount that can be added to the account",
    "transaction_charged_twice": "the same payment appears twice",
    "request_refund": "ask for money back for a purchase",
    "edit_personal_details": "update name, address or other personal information",
}
missing = [k for k in INTENTS4 if k not in B77_LABELS]
if missing: log.info(f"exp4: intents missing from banking77, removed: {missing}")
INTENTS4 = {k: v for k, v in INTENTS4.items() if k in B77_LABELS}
I4 = list(INTENTS4)
rng4 = np.random.default_rng(CFG["seed"] + 4)
CODES = {lab: f"K{c}" for lab, c in zip(I4, rng4.choice(np.arange(10, 100), len(I4), replace=False))}
shift = rng4.integers(1, len(I4))
MISLEAD = {lab: humanize(I4[(j + shift) % len(I4)]) for j, lab in enumerate(I4)}  # no key points to its own intent

def criteria_for(cond, order):
    """Returns {shown_key: description} and {shown_key: true intent of the description}."""
    crit, key2int = {}, {}
    for lab in order:
        key = dict(original=humanize(lab), name_only=humanize(lab), code_desc=CODES[lab],
                   code_only=CODES[lab], misleading=MISLEAD[lab])[cond]
        crit[key] = None if cond in ("name_only", "code_only") else INTENTS4[lab]
        key2int[key] = lab
    return crit, key2int

INSTR4 = "Which option matches the customer's request?"
if RUN["exp4"]:
    L4 = JsonlLog("exp4_labels")
    sub = b77[b77.label_text.isin(I4)].groupby("label_text", group_keys=False).apply(
        lambda g: g.sample(n=min(CFG["exp4_per_intent"], len(g)), random_state=CFG["seed"])).reset_index(drop=True)
    for i, row in sub.iterrows():
        order = list(np.random.default_rng(CFG["seed"] * 4000 + i).permutation(I4))  # same order in the 5 conditions
        for cond in ["original", "name_only", "code_desc", "code_only", "misleading"]:
            k = f"{i}|{cond}"
            if k in L4.done: continue
            crit, key2int = criteria_for(cond, order)
            qs = choice_q(crit, INSTR4); bud = fit_budget(row["text"], qs)
            a = agent.predict(row["text"], qs, **bud)["answers"]["q"]
            pred_int = key2int[a["choice"]]
            L4.write(dict(key=k, idx=i, condition=cond, gold=row["label_text"], pred_key=a["choice"],
                          pred_intent=pred_int, correct=pred_int == row["label_text"],
                          followed_key=(cond == "misleading" and a["choice"] == humanize(row["label_text"])),
                          p_max=max(a["probabilities"].values()), **bud, **seq_stats(row["text"], qs, **bud)))
    d4 = L4.df()
    s4 = d4.groupby("condition").agg(n=("idx", "count"), accuracy=("correct", "mean"),
                                     followed_key=("followed_key", "mean"), p_max=("p_max", "mean"))
    s4 = s4.reindex(["original", "name_only", "code_desc", "code_only", "misleading"])
    s4["chance"] = 1 / len(I4)
    s4.to_csv(OUT / "results" / f"exp4_summary{SFX}.csv"); display(s4)

    plt.figure(figsize=(8, 4))
    plt.bar(s4.index, s4.accuracy, label="correct by description")
    plt.bar(["misleading"], s4.loc[["misleading"], "followed_key"], bottom=s4.loc[["misleading"], "accuracy"],
            color="C3", label="followed the misleading key")
    plt.axhline(1 / len(I4), ls="--", c="k", label="chance"); plt.ylim(0, 1); plt.legend()
    plt.title("what Laya uses to decide: key or description?")
    savefig("exp4_labels")

In [ ]:
# 8. Experiment 6: X-ray of the vectors at the [MASK] markers
if RUN["exp6"]:
    from transformers import AutoModel
    import umap
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score, GroupKFold
    from sklearn.metrics import silhouette_score

    base = AutoModel.from_pretrained("answerdotai/ModernBERT-large", attn_implementation="sdpa").to(DEVICE).eval()
    assert base.config.vocab_size >= len(TOK), "the base ModernBERT vocabulary does not match Laya's"
    M = agent.model
    sub = b77[b77.label_text.isin(I4)].groupby("label_text", group_keys=False).apply(
        lambda g: g.sample(n=min(CFG["exp6_per_intent"], len(g)), random_state=CFG["seed"] + 6)).reset_index(drop=True)

    V = {"base": [], "laya_encoder": [], "laya_head": []}; meta6 = []; agree = []
    amp = torch.autocast("cuda", dtype=torch.float16) if DEVICE == "cuda" else torch.autocast("cpu", enabled=False)
    for i, row in sub.iterrows():
        order = list(np.random.default_rng(CFG["seed"] * 6000 + i).permutation(I4))  # decouples identity from position
        crit, key2int = criteria_for("original", order)
        qs = choice_q(crit, INSTR4); bud = fit_budget(row["text"], qs)
        ids, markers = build_sequence(TOK, row["text"], Agent._to_internal(qs["q"]), **bud)
        x = torch.tensor([ids], device=DEVICE); att = torch.ones_like(x); mk = torch.tensor(markers, device=DEVICE)
        with torch.no_grad(), amp:
            hb = base(input_ids=x, attention_mask=att).last_hidden_state[0, mk]
            he = M.encoder(input_ids=x, attention_mask=att).last_hidden_state
            h = he + M.type_emb(torch.tensor([QTYPES["choice"]], device=DEVICE))[:, None, :]
            for layer in M.head.layers:
                h = layer(h, src_key_padding_mask=~att.bool())
            hh = h[0, mk]; logits = M.scorer(hh).squeeze(-1).float()
        pred_manual = order[int(logits.argmax())]
        pred_api = key2int[agent.predict(row["text"], qs, **bud)["answers"]["q"]["choice"]]
        agree.append(pred_manual == pred_api)
        for name, t in [("base", hb), ("laya_encoder", he[0, mk]), ("laya_head", hh)]:
            V[name].append(t.float().cpu().numpy())
        for pos, lab in enumerate(order):
            meta6.append(dict(example=i, option=lab, position=pos, is_gold=lab == row["label_text"],
                              gold=row["label_text"], pred=pred_manual))
    log.info(f"exp6: manual extraction agrees with agent.predict on {np.mean(agree):.1%} of texts")
    meta6 = pd.DataFrame(meta6)
    X = {k: np.concatenate(v) for k, v in V.items()}
    np.savez_compressed(OUT / "results" / f"exp6_marker_vectors{SFX}.npz", **X)
    meta6.to_csv(OUT / "results" / f"exp6_marker_meta{SFX}.csv", index=False)

    rows, Z = [], {}
    y, grp = meta6.is_gold.values, meta6.example.values
    for name, A in X.items():
        An = A / np.linalg.norm(A, axis=1, keepdims=True)
        auroc = cross_val_score(LogisticRegression(max_iter=2000, class_weight="balanced"), An, y,
                                groups=grp, cv=GroupKFold(5), scoring="roc_auc").mean()
        rows.append(dict(space=name, probe_auroc_is_gold=auroc,
                         silhouette_by_option=silhouette_score(An, meta6.option, metric="cosine"),
                         silhouette_by_is_gold=silhouette_score(An, y, metric="cosine")))
        Z[name] = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="cosine", random_state=CFG["seed"]).fit_transform(An)
    s6 = pd.DataFrame(rows).set_index("space")
    s6.to_csv(OUT / "results" / f"exp6_summary{SFX}.csv"); display(s6)
    np.savez_compressed(OUT / "results" / f"exp6_umap{SFX}.npz", **Z)

    fig, ax = plt.subplots(2, 3, figsize=(16, 10))
    cmap = {lab: plt.cm.tab20(j) for j, lab in enumerate(I4)}
    titles = {"base": "untrained ModernBERT-large", "laya_encoder": "Laya encoder", "laya_head": "after the 2 extra layers"}
    for j, name in enumerate(X):
        z = Z[name]
        ax[0, j].scatter(z[:, 0], z[:, 1], s=3, c=[cmap[o] for o in meta6.option])
        ax[0, j].set_title(f"{titles[name]}\ncolor = which option")
        ax[1, j].scatter(z[~y, 0], z[~y, 1], s=3, c="lightgray")
        ax[1, j].scatter(z[y, 0], z[y, 1], s=6, c="C3")
        ax[1, j].set_title(f"red = correct option for the text (AUROC {s6.loc[name, 'probe_auroc_is_gold']:.2f})")
        for a in ax[:, j]: a.set_xticks([]); a.set_yticks([])
    savefig("exp6_xray")

In [ ]:
# 5b. Experiment 2b: does the position bias come from the distance in tokens to the customer's text?
# Design: 50 options, the correct one placed at 11 fixed positions. Two layouts:
#   "short" = option name only; "long" = name + the same neutral description for all (longer, no extra information).
# At the same position in the list, the long layout puts the option farther from the text. If accuracy follows the
# distance in tokens (and not the position in the list), the bias comes from ModernBERT's local attention.
if RUN.get("exp2b", True):
    import statsmodels.formula.api as smf
    ec = agent.model.encoder.config
    WIN = getattr(ec, "local_attention", None)
    arch = dict(local_attention=WIN, global_every=getattr(ec, "global_attn_every_n_layers", None),
                layers=getattr(ec, "num_hidden_layers", None))
    log.info(f"exp2b: encoder architecture {arch}")
    N2B, POS2B = 50, [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 49]
    LAYOUTS = {"short": None, "long": "banking intent"}
    L2b = JsonlLog("exp2b_distance")
    ex = sample_b77(5 if QUICK else 100, CFG["seed"] + 21)
    for i, row in ex.iterrows():
        gold = humanize(row["label_text"])
        rng = np.random.default_rng(CFG["seed"] * 5000 + i)
        distract = [o for o in option_set(gold, N2B, rng) if o != gold]  # the distractors always keep the same order
        for layout, desc in LAYOUTS.items():
            for p in POS2B:
                k = f"{layout}|{p}|{i}"
                if k in L2b.done: continue
                order = distract[:p] + [gold] + distract[p:]
                qs = choice_q({o: desc for o in order}, INSTR)
                bud = fit_budget(row["text"], qs)
                ids, markers = build_sequence(TOK, row["text"], Agent._to_internal(qs["q"]), **bud)
                seps = [j for j, t in enumerate(ids) if t == TOK.sep_token_id]
                a = agent.predict(row["text"], qs, **bud)["answers"]["q"]
                probs = a["probabilities"]
                L2b.write(dict(key=k, layout=layout, pos=p, idx=i, gold=gold, pred=a["choice"],
                               correct=a["choice"] == gold, p_gold=probs[gold],
                               rank_gold=sorted(probs.values(), reverse=True).index(probs[gold]) + 1,
                               dist_tokens=(seps[1] + 1) - markers[p], seq_len=len(ids), **bud))
        if i % 20 == 0: log.info(f"exp2b: {i + 1}/{len(ex)} texts")

    d = L2b.df()
    s_pos = d.groupby(["layout", "pos"]).agg(dist_tokens=("dist_tokens", "mean"), accuracy=("correct", "mean"),
                                             p_gold=("p_gold", "mean"), rank_gold=("rank_gold", "mean"),
                                             seq_len=("seq_len", "mean"))
    d["dist_bin"] = pd.cut(d.dist_tokens, range(0, int(d.dist_tokens.max()) + 33, 32), right=False)
    s_bin = d.groupby(["layout", "dist_bin"], observed=True).agg(n=("correct", "size"), accuracy=("correct", "mean"),
                                                                 p_gold=("p_gold", "mean"))
    # position and distance together in one model: which of the two explains accuracy?
    d["y"] = d.correct.astype(int); d["pos_std"] = d.pos / 49; d["dist_std"] = d.dist_tokens / 100
    try:  # with few texts (QUICK) the fit may not converge; it must not break the cell
        fit = smf.logit("y ~ pos_std + dist_std", data=d).fit(disp=0, cov_type="cluster", cov_kwds={"groups": d.idx})
        coefs = pd.DataFrame({"coef": fit.params, "p_value": fit.pvalues})
    except Exception as e:
        log.info(f"exp2b: logistic regression failed ({e!r})"); coefs = pd.DataFrame()
    s_pos.to_csv(OUT / "results" / f"exp2b_by_position{SFX}.csv")
    s_bin.to_csv(OUT / "results" / f"exp2b_by_distance{SFX}.csv")
    coefs.to_csv(OUT / "results" / f"exp2b_logit{SFX}.csv")
    json.dump(arch, open(OUT / "results" / f"exp2b_encoder_arch{SFX}.json", "w"))
    display(s_pos); display(coefs)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for layout, g in s_pos.reset_index().groupby("layout"):
        ax[0].plot(g.pos, g.accuracy, "o-", label=layout)
        ax[1].plot(g.dist_tokens, g.accuracy, "o-", label=layout)
    ax[0].set_xlabel("position of the correct option in the list (0 = first)")
    ax[1].set_xlabel("distance in tokens from the correct option to the customer's text")
    if WIN:
        for x, lab in [(WIN // 2, "local reach"), (WIN, "local window")]:
            ax[1].axvline(x, ls="--", c="gray"); ax[1].text(x, 0.02, f" {lab} ({x})", color="gray")
    for a in ax: a.set_ylabel("accuracy"); a.set_ylim(0, 1); a.legend()
    savefig("exp2b_distance")

In [ ]:
# 6b. Experiment 3b: expanded minimal-edit pairs
# Each sentence template is filled 3 times (6 categories × 4 templates × 3 = 72 pairs that should change,
# plus 15 paraphrase controls). Each pair runs with the options in 2 orders.
# Format: (category, kind, text_a, text_b, option_A, option_B, (correct_a, correct_b), fillers)
T3B = [
    ("count", "change", "I was charged once for my {0}.", "I was charged twice for my {0}.",
     ("single charge", "a question about one normal charge"), ("duplicate charge", "the same purchase was billed more than once"),
     ("A", "B"), [("coffee",), ("taxi ride",), ("hotel booking",)]),
    ("count", "change", "I entered the wrong PIN once {0}.", "I entered the wrong PIN three times {0}.",
     ("pin question", "a general question about the PIN"), ("card blocked", "the card is blocked after too many wrong PIN attempts"),
     ("A", "B"), [("at the ATM",), ("at a shop",), ("at the petrol station",)]),
    ("count", "change", "I paid for one {0} and received one.", "I paid for two {1} and received one.",
     ("order complete", "everything that was paid for was delivered"), ("missing item", "fewer items were delivered than were paid for"),
     ("A", "B"), [("ticket", "tickets"), ("shirt", "shirts"), ("book", "books")]),
    ("count", "change", "My {0} was paid into my account once this month.", "My {0} was paid into my account twice this month.",
     ("normal payment", "an expected single payment was received"), ("duplicate payment", "the same payment was received more than once"),
     ("A", "B"), [("salary",), ("pension",), ("allowance",)]),
    ("negation", "change", "I want to cancel my {0}.", "I do not want to cancel my {0}.",
     ("cancel", "the customer wants to end it"), ("keep", "the customer wants to keep it"),
     ("A", "B"), [("subscription",), ("membership",), ("plan",)]),
    ("negation", "change", "The {0} arrived.", "The {0} never arrived.",
     ("delivered", "the item was delivered"), ("delivery problem", "the item has not been delivered"),
     ("A", "B"), [("package",), ("card",), ("order",)]),
    ("negation", "change", "I recognise this {0} on my statement.", "I don't recognise this {0} on my statement.",
     ("confirmed", "the customer confirms it is theirs"), ("fraud report", "the customer did not make it"),
     ("A", "B"), [("payment",), ("transfer",), ("charge",)]),
    ("negation", "change", "The {0} works after the update.", "The {0} no longer works after the update.",
     ("working fine", "it works as expected"), ("bug", "it is broken or not working"),
     ("A", "B"), [("app",), ("website",), ("card reader",)]),
    ("numbers", "change", "My balance is ${0} and I want to withdraw ${1}.", "My balance is ${1} and I want to withdraw ${0}.",
     ("sufficient funds", "the balance covers the withdrawal"), ("insufficient funds", "the withdrawal is larger than the balance"),
     ("A", "B"), [("500", "200"), ("900", "300"), ("1000", "50")]),
    ("numbers", "change", "The invoice says ${0} and I was charged ${0}.", "The invoice says ${0} and I was charged ${1}.",
     ("amount correct", "charged exactly the invoice amount"), ("amount mismatch", "charged a different amount than the invoice"),
     ("A", "B"), [("120", "210"), ("45", "54"), ("310", "130")]),
    ("numbers", "change", "I ordered size {0} and received size {0}.", "I ordered size {0} and received size {1}.",
     ("correct item", "received exactly the size ordered"), ("wrong item", "received a different size than ordered"),
     ("A", "B"), [("42", "24"), ("38", "83"), ("46", "64")]),
    ("numbers", "change", "My flight leaves at {0} and I will get to the airport at {1}.", "My flight leaves at {1} and I will get to the airport at {0}.",
     ("on time", "the customer will make the flight"), ("missed flight", "the customer will miss the flight"),
     ("A", "B"), [("9", "7"), ("11", "8"), ("6", "4")]),
    ("roles", "change", "{1} sent money to me.", "I sent money to {0}.",
     ("incoming transfer", "the customer received money"), ("outgoing transfer", "the customer sent money"),
     ("A", "B"), [("the bank", "The bank"), ("my landlord", "My landlord"), ("my friend", "My friend")]),
    ("roles", "change", "I refunded the {0}.", "The {0} refunded me.",
     ("refund issued", "the writer gave money back"), ("refund received", "the writer got money back"),
     ("A", "B"), [("customer",), ("shop",), ("seller",)]),
    ("roles", "change", "I owe {0} money.", "{1} owes me money.",
     ("writer owes", "the writer has a debt to someone"), ("owed to writer", "someone has a debt to the writer"),
     ("A", "B"), [("my brother", "My brother"), ("the shop", "The shop"), ("my landlord", "My landlord")]),
    ("roles", "change", "{1} paid me.", "I paid {0}.",
     ("money received", "the writer received a payment"), ("money paid", "the writer made a payment"),
     ("A", "B"), [("my employer", "My employer"), ("my tenant", "My tenant"), ("my client", "My client")]),
    ("time", "change", "I want to pay before {0}.", "I want to pay after {0}.",
     ("on-time payment", "paying before the deadline"), ("late payment", "paying after the deadline"),
     ("A", "B"), [("the due date",), ("the deadline",), ("the end of the month",)]),
    ("time", "change", "I will travel {0} next week.", "I travelled {0} last week.",
     ("future trip", "about a trip that has not happened yet"), ("past trip", "about a trip that already happened"),
     ("A", "B"), [("abroad",), ("to Spain",), ("to Japan",)]),
    ("time", "change", "I want to stop a {0} that will go out tomorrow.", "I want to stop a {0} that went out yesterday.",
     ("stop future payment", "cancel a payment before it is sent"), ("reverse past payment", "get back a payment that was already sent"),
     ("A", "B"), [("payment",), ("transfer",), ("direct debit",)]),
    ("time", "change", "My {0} renews next month.", "My {0} renewed last month.",
     ("upcoming renewal", "a renewal that has not happened yet"), ("past renewal", "a renewal that already happened"),
     ("A", "B"), [("subscription",), ("membership",), ("insurance",)]),
    ("quantifier", "change", "All of my {0} failed today.", "None of my {0} failed today.",
     ("failed", "they did not go through"), ("went through", "they went through normally"),
     ("A", "B"), [("transfers",), ("payments",), ("top-ups",)]),
    ("quantifier", "change", "Every {0} I tried was declined.", "No {0} I tried was declined.",
     ("declined", "the attempts were rejected"), ("accepted", "the attempts were accepted"),
     ("A", "B"), [("card payment",), ("withdrawal",), ("transfer",)]),
    ("quantifier", "change", "All of my {0} were refunded.", "None of my {0} were refunded.",
     ("refund complete", "the money was returned"), ("refund missing", "the money was not returned"),
     ("A", "B"), [("orders",), ("purchases",), ("tickets",)]),
    ("quantifier", "change", "I received all of my {0}.", "I received none of my {0}.",
     ("all delivered", "everything arrived"), ("nothing delivered", "nothing arrived"),
     ("A", "B"), [("parcels",), ("cards",), ("documents",)]),
    # controls: the edit does not change the meaning, so the answer should not change
    ("paraphrase", "same", "I was charged twice for my {0}.", "I was charged two times for my {0}.",
     ("single charge", "a question about one normal charge"), ("duplicate charge", "the same purchase was billed more than once"),
     ("B", "B"), [("coffee",), ("taxi ride",), ("hotel booking",)]),
    ("paraphrase", "same", "I do not want to cancel my {0}.", "I don't want to cancel my {0}.",
     ("cancel", "the customer wants to end it"), ("keep", "the customer wants to keep it"),
     ("B", "B"), [("subscription",), ("membership",), ("plan",)]),
    ("paraphrase", "same", "My balance is ${1} and I want to withdraw ${0}.", "My balance is {1} dollars and I want to withdraw {0} dollars.",
     ("sufficient funds", "the balance covers the withdrawal"), ("insufficient funds", "the withdrawal is larger than the balance"),
     ("B", "B"), [("500", "200"), ("900", "300"), ("1000", "50")]),
    ("paraphrase", "same", "The {0} never arrived.", "The {0} has not arrived.",
     ("delivered", "the item was delivered"), ("delivery problem", "the item has not been delivered"),
     ("B", "B"), [("package",), ("card",), ("order",)]),
    ("paraphrase", "same", "I sent money to {0}.", "I transferred money to {0}.",
     ("incoming transfer", "the customer received money"), ("outgoing transfer", "the customer sent money"),
     ("B", "B"), [("the bank",), ("my landlord",), ("my friend",)]),
]
OTHER3B, INSTR3B = ("other", "anything else"), "Which option best describes the customer's message?"

if RUN.get("exp3b", True):
    L3b = JsonlLog("exp3b_minimal_pairs")
    for t_id, (cat, kind, ta_t, tb_t, oA, oB, (ga, gb), fills) in enumerate(T3B):
        opt = {"A": oA, "B": oB}
        for f_id, f in enumerate(fills):
            ta, tb = ta_t.format(*f), tb_t.format(*f)
            for order_name, seq in [("fwd", ["A", "B"]), ("rev", ["B", "A"])]:
                k = f"{t_id}|{f_id}|{order_name}"
                if k in L3b.done: continue
                crit = {opt[s][0]: opt[s][1] for s in seq}; crit[OTHER3B[0]] = OTHER3B[1]
                qs = choice_q(crit, INSTR3B)
                bud = fit_budget(max(ta, tb, key=len), qs)
                A = agent.predict(ta, qs, **bud)["answers"]["q"]; B = agent.predict(tb, qs, **bud)["answers"]["q"]
                gA, gB = opt[ga][0], opt[gb][0]
                L3b.write(dict(key=k, template=t_id, filler=f_id, category=cat, kind=kind, order=order_name,
                               text_a=ta, text_b=tb, gold_a=gA, gold_b=gB, pred_a=A["choice"], pred_b=B["choice"],
                               correct_a=A["choice"] == gA, correct_b=B["choice"] == gB,
                               flipped=A["choice"] != B["choice"], tv=tv(A["probabilities"], B["probabilities"]),
                               p_gold_b_in_a=A["probabilities"][gB], p_gold_b_in_b=B["probabilities"][gB],
                               probs_a=A["probabilities"], probs_b=B["probabilities"]))
    d = L3b.df()
    d["both_correct"] = d.correct_a & d.correct_b
    d["shift_to_gold_b"] = d.p_gold_b_in_b - d.p_gold_b_in_a
    d["moved_right_way"] = d.shift_to_gold_b > 0
    agg = dict(runs=("key", "count"), acc_a=("correct_a", "mean"), acc_b=("correct_b", "mean"),
               both_correct=("both_correct", "mean"), flip_rate=("flipped", "mean"), mean_tv=("tv", "mean"),
               shift_to_gold_b=("shift_to_gold_b", "mean"), moved_right_way=("moved_right_way", "mean"))
    s3b = d.groupby(["kind", "category"]).agg(**agg)
    s3b_t = d.groupby(["kind", "category", "template"]).agg(**agg)
    s3b_t["example_a"] = d.groupby(["kind", "category", "template"]).text_a.first()
    s3b.to_csv(OUT / "results" / f"exp3b_summary{SFX}.csv")
    s3b_t.to_csv(OUT / "results" / f"exp3b_by_template{SFX}.csv")
    display(s3b)

    cols = ["acc_a", "acc_b", "both_correct", "flip_rate"]
    g = s3b.reset_index().set_index("category")[cols]
    g.plot.bar(figsize=(11, 4), rot=0, ylim=(0, 1))
    plt.title("minimal edits: accuracy on each side, on both sides, and flip rate (paraphrase = control)")
    savefig("exp3b_minimal_pairs")

In [ ]:
# 9. Package the results
zpath = ROOT / f"laya_bench_{RUN_ID}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(ROOT))
log.info(f"done: {zpath}")